# 01 — QC, revue et splits groupés (tâches 06 à 08)


Le fichier de revue est écrit avant le contrôle bloquant. Les exclusions validées alimentent directement le split.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import experiment_config as cfg
from src.io.database_h5 import load_nir_uco_h5
from src.workflows.protocol_split import (
    build_protocol_manifest,
    build_split_diagnostics,
)
from src.workflows.quality_check import (
    add_robust_spectral_qc,
    apply_qc_reviews,
    build_image_qc_table,
    build_image_qc_warnings,
    build_object_qc_table,
    build_object_qc_warnings,
    build_object_shape_check_tables,
    build_qc_alerts_table,
    build_qc_exclusion_report,
    build_qc_protocol,
    build_qc_visual_review_report,
    check_missing_required_fields,
    merge_existing_reviews_or_initialize,
    validate_qc_review_closure,
)

H5_PATH = PROJECT_ROOT.joinpath(*cfg.DATABASE_H5_RELATIVE_PATH)
RESULTS_DIR = PROJECT_ROOT.joinpath(*cfg.QC_RESULTS_RELATIVE_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = {
    key: RESULTS_DIR / filename for key, filename in cfg.QC_OUTPUT_FILENAMES.items()
}
VERSIONED_REVIEW = PROJECT_ROOT.joinpath(*cfg.QC_REVIEW_DECISIONS_RELATIVE_PATH)
object_db, image_db = load_nir_uco_h5(
    H5_PATH, reconstruct_heavy_object_arrays=True
)


In [2]:
image_qc = build_image_qc_table(image_db)
object_qc = build_object_qc_table(
    object_db,
    image_db=image_db,
    border_margin=cfg.QC_BORDER_MARGIN,
)
object_qc = add_robust_spectral_qc(object_qc, object_db)
image_warnings = build_image_qc_warnings(image_qc)
object_warnings = build_object_qc_warnings(object_qc)
missing_fields = check_missing_required_fields(image_db, object_db)
_, bad_shapes = build_object_shape_check_tables(object_db, image_db)
alerts = build_qc_alerts_table(
    image_warnings,
    object_warnings,
    missing_fields,
    bad_shapes,
)
image_qc.to_parquet(OUTPUT["image_summary"], index=False)
object_qc.to_parquet(OUTPUT["object_summary"], index=False)
alerts.to_parquet(OUTPUT["alerts"], index=False)


In [3]:
review = merge_existing_reviews_or_initialize(alerts, VERSIONED_REVIEW)
if OUTPUT["review"].exists():
    review = merge_existing_reviews_or_initialize(review_source=OUTPUT["review"], qc_alerts_df=alerts)
review.to_parquet(OUTPUT["review"], index=False)
build_qc_visual_review_report(
    alerts,
    object_db,
    image_db,
    object_qc,
    OUTPUT["visual_review"],
)
review

,record_type,record_id,flag_type,review_status,review_decision,reviewer,review_date,review_comment,review_evidence
0,object,alm1pea1_obj030,possible_merged_object,reviewed,accept_as_is,visual_review,2026-07-29,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
1,object,alm3pea3_obj005,possible_merged_object,reviewed,accept_as_is,visual_review,2026-07-29,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
2,object,alm3pea3_obj029,possible_merged_object,reviewed,accept_as_is,visual_review,2026-07-29,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
3,object,pea3_pos3_obj004,possible_merged_object,reviewed,accept_as_is,visual_review,2026-07-29,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
4,object,pea3_pos3_obj008,possible_merged_object,reviewed,accept_as_is,visual_review,2026-07-29,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
5,object,almond2_obj032,possible_merged_object,reviewed,accept_as_is,visual_review,2026-07-29,"Visual inspection of the source image, object ...",qc_visual_review_report.pdf
6,object,alm3pea2_obj026,robust_spectral_outlier,reviewed,accept_as_is,visual_review,2026-07-29,Ignoring for now,qc_visual_review_report.pdf
7,object,alm3pea4_obj008,robust_spectral_outlier,reviewed,accept_as_is,visual_review,2026-07-29,Ignoring for now,qc_visual_review_report.pdf


In [4]:
# Point de contrôle humain: le fichier pending et le PDF existent déjà.
validate_qc_review_closure(review)
resolved_alerts = apply_qc_reviews(alerts, review, require_complete=True)

In [5]:
exclusions = build_qc_exclusion_report(resolved_alerts)
qc_protocol = build_qc_protocol(alerts, review, exclusions)
split_manifest, split_checks = build_protocol_manifest(
    image_db,
    object_db,
    exclusion_manifest=exclusions,
    strict=True,
)
split_diagnostics = build_split_diagnostics(split_manifest, object_db)
exclusions.to_parquet(OUTPUT["exclusion_manifest"], index=False)
qc_protocol.to_parquet(OUTPUT["protocol"], index=False)
split_manifest.to_parquet(OUTPUT["split_manifest"], index=False)
split_diagnostics.to_parquet(OUTPUT["split_diagnostics"], index=False)
qc_protocol


,protocol_version,qc_policy_hash,alerts_hash,review_hash,n_alerts,n_pending,n_excluded,closure_status
0,8tracks_v1,978faa83e2349368c03e950de7ea938837c6d0912584b4...,ef59d9469c50b826ca6b43366ba10e5e1539579bf7d470...,7042c7218ea93c7a180f4433535e18a0b39bcec24d8845...,8,0,0,closed
